# Copilot Audit Log — Direct Ingester (Fabric)

End-to-end ingestion notebook that **calls Microsoft Graph directly** from inside Fabric, parses the audit-log records on the fly, and writes a Delta table the **AI-in-One Dashboard** and **AI Business Value Dashboard** PBITs consume directly.

**No PowerShell, no intermediate CSVs, no separate parser.** Replaces the previous flow:

```
OLD:  PowerShell script → local CSV → manual move into Files/audit_raw/ → Copilot_Audit_Log_Parser.ipynb → Delta
NEW:  This notebook (Graph → parse → Delta)
```

**Output**: Lakehouse Delta table `Copilot_Interactions_Parsed` (same name + schema as the existing parser, so the PBIT works without changes).

**Schedule**: weekly (matches Microsoft Graph's 7-day audit-log query window). Use the **Schedule** button at the top of the notebook.

**Permissions**: app registration with `AuditLogsQuery.Read.All` (Application permission, admin-consented). No `Sites.Selected` needed — this path doesn't touch SharePoint.


## 1. Configuration

Fill in the four config values below. The client secret should ideally come from a Key Vault — see the commented alternative.

**Lakehouse Schemas note**: the default `OUTPUT_TABLE` is `dbo.Copilot_Interactions_Parsed`. This works for both schema-enabled Lakehouses (the new Fabric default) and non-schema Lakehouses — `dbo` is the implicit default schema either way. If your Lakehouse uses a different schema, change the prefix.


In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'                  # Entra → Overview → Tenant ID
CLIENT_ID     = '<your-app-reg-client-id>'            # Entra → App registrations → your app → Application (client) ID
CLIENT_SECRET = '<your-client-secret-value>'          # The secret VALUE (not Secret ID). Prefer Key Vault — see below.

LOOKBACK_DAYS = 7                                     # Graph caps audit-log queries at 7 days back per request.
OUTPUT_TABLE  = 'dbo.Copilot_Interactions_Parsed'     # Delta table consumed by the PBIT (schema-qualified for Lakehouse Schemas mode)
WRITE_MODE    = 'overwrite'                           # 'overwrite' for full snapshots; 'append' for incremental

# --- Large-tenant tuning (prevents timeouts on big audit volumes) -------------------
CHUNK_HOURS             = 24    # split the lookback into windows this size; smaller = faster per async query & more resumable
MAX_WAIT_MIN_PER_QUERY  = 240   # generous ceiling per async audit query (Purview can take well over 30 min on large tenants)
POLL_INTERVAL_SEC       = 30    # how often to poll an async query for completion
STAGING_DIR             = 'Files/_audit_staging'  # Lakehouse Files path; records are streamed here page-by-page (bounded memory)

# --- Production: load secret from Key Vault instead of hardcoding -------------------
# from notebookutils.credentials import getSecret
# CLIENT_SECRET = getSecret('https://<your-vault>.vault.azure.net/', 'CopilotAuditAppSecret')


## 2. Authenticate to Microsoft Graph

Service principal (client secret) flow — same auth pattern as the PowerShell scripts.


In [ ]:
import requests, time

# Auto-refreshing app-only Graph token. Long (multi-hour) chunked runs would otherwise
# fail when the 1-hour token expires mid-pull, so we transparently refresh every ~50 min.
def _get_graph_token() -> str:
    url  = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'
    body = {
        'client_id':     CLIENT_ID,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': CLIENT_SECRET,
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

_token = {'value': None, 'ts': 0.0}

def get_headers() -> dict:
    if _token['value'] is None or (time.time() - _token['ts']) > 50 * 60:
        _token['value'] = _get_graph_token()
        _token['ts'] = time.time()
    return {'Authorization': f'Bearer {_token["value"]}', 'Content-Type': 'application/json'}

get_headers()
print('✓ Graph token acquired (auto-refreshing).')


## 3. Build query windows + the `create_query` helper

The lookback is split into smaller time windows (`CHUNK_HOURS`). Each window is its own async
audit query, which completes far faster than one giant query — the main cause of timeouts on
large tenants — and lets the run checkpoint window-by-window.

In [ ]:
from datetime import datetime, timezone, timedelta

end_date   = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

# Split the lookback into smaller windows so each async audit query is smaller and completes
# faster. One big query over a high-volume range is the main cause of the 'times out on large
# data sets' symptom.
windows = []
_cur = start_date
while _cur < end_date:
    _nxt = min(_cur + timedelta(hours=CHUNK_HOURS), end_date)
    windows.append((_cur, _nxt))
    _cur = _nxt
print(f'✓ Lookback {LOOKBACK_DAYS}d split into {len(windows)} window(s) of {CHUNK_HOURS}h.')

def create_query(win_start, win_end) -> str:
    body = {
        'displayName':         f'Copilot Interactions {win_start:%Y%m%d%H%M}-{win_end:%Y%m%d%H%M}',
        'filterStartDateTime': win_start.isoformat(),
        'filterEndDateTime':   win_end.isoformat(),
        'recordTypeFilters':   ['copilotInteraction'],
        'operationFilters':    [],
    }
    r = requests.post(
        'https://graph.microsoft.com/beta/security/auditLog/queries',
        json=body, headers=get_headers(), timeout=30,
    )
    r.raise_for_status()
    return r.json()['id']


## 4. Resumable wait helper

Polls an async query until it succeeds, with a **generous, configurable** ceiling
(`MAX_WAIT_MIN_PER_QUERY`) instead of a hard 30-minute cut-off. The token auto-refreshes while
waiting, so multi-hour jobs don't fail on expiry.

In [ ]:
import time

def wait_for_query(query_id: str, max_wait_min: int = MAX_WAIT_MIN_PER_QUERY) -> None:
    started  = time.time()
    deadline = started + max_wait_min * 60
    polls = 0
    while time.time() < deadline:
        polls += 1
        r = requests.get(
            f'https://graph.microsoft.com/beta/security/auditLog/queries/{query_id}',
            headers=get_headers(), timeout=30,
        )
        r.raise_for_status()
        status = r.json()['status']
        if status == 'succeeded':
            return
        if status in ('failed', 'cancelled'):
            raise RuntimeError(f'Query {query_id} ended with status: {status}')
        if polls % 4 == 0:
            mins = int((time.time() - started) // 60)
            print(f'    … still {status} ({mins}m elapsed, {polls} polls)')
        time.sleep(POLL_INTERVAL_SEC)
    raise TimeoutError(
        f'Query {query_id} did not succeed within {max_wait_min} min. '
        f'Increase MAX_WAIT_MIN_PER_QUERY or reduce CHUNK_HOURS.')


## 5. Fetch records → stream to Lakehouse Files (chunked, bounded memory)

Each page of results is written straight to Lakehouse **Files** as JSON-lines, so the full
result set is **never** held in driver memory — the other main cause of OOM/timeouts on large
tenants. Set `WRITE_MODE = 'overwrite'` to clear the staging area first.

In [ ]:
import os, json

# Stream every page straight to Lakehouse Files as JSON-lines so we never accumulate the full
# result set in the driver. STAGING_DIR is relative to the attached default lakehouse.
STAGING_ABS = '/lakehouse/default/' + STAGING_DIR.rstrip('/')
os.makedirs(STAGING_ABS, exist_ok=True)
if WRITE_MODE == 'overwrite':
    for _fn in os.listdir(STAGING_ABS):
        try:
            os.remove(os.path.join(STAGING_ABS, _fn))
        except OSError:
            pass

def _row(r: dict) -> dict:
    # Re-serialise AuditData to a string, matching the schema the parser expects downstream.
    return {
        'RecordId':                  r.get('id'),
        'CreationDate':              r.get('createdDateTime'),
        'RecordType':                r.get('auditLogRecordType'),
        'Operation':                 r.get('operation'),
        'AuditData':                 json.dumps(r.get('auditData', {})) if r.get('auditData') else None,
        'AssociatedAdminUnits':      json.dumps(r.get('associatedAdminUnits', [])),
        'AssociatedAdminUnitsNames': json.dumps(r.get('associatedAdminUnitsNames', [])),
    }

page_idx = 0
total    = 0
for wi, (ws, we) in enumerate(windows, 1):
    qid = create_query(ws, we)
    print(f'[{wi}/{len(windows)}] {ws:%Y-%m-%d %H:%M} → {we:%Y-%m-%d %H:%M}  query {qid}')
    wait_for_query(qid)
    next_url = (f'https://graph.microsoft.com/beta/security/auditLog/queries/'
                f'{qid}/records?$top=999')
    while next_url:
        r = requests.get(next_url, headers=get_headers(), timeout=60)
        r.raise_for_status()
        data = r.json()
        vals = data.get('value', [])
        if vals:
            fp = os.path.join(STAGING_ABS, f'page_{page_idx:06d}.jsonl')
            with open(fp, 'w', encoding='utf-8') as f:
                for rec in vals:
                    f.write(json.dumps(_row(rec)) + '\n')
            page_idx += 1
            total    += len(vals)
        next_url = data.get('@odata.nextLink')
    print(f'    cumulative records: {total:,}')

print(f'✓ Streamed {total:,} records to {STAGING_DIR} across {page_idx} page file(s).')


## 6. Read the staged records (distributed)

Load the staged JSON-lines with Spark — a distributed read, not a driver-side
`createDataFrame` from a giant Python list.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

_cols = ['RecordId', 'CreationDate', 'RecordType', 'Operation',
         'AuditData', 'AssociatedAdminUnits', 'AssociatedAdminUnitsNames']

if page_idx == 0:
    # No records in range — build an empty frame with the expected schema so the rest runs.
    _empty = StructType([StructField(c, StringType()) for c in _cols])
    raw = spark.createDataFrame([], _empty)
else:
    raw = spark.read.json(STAGING_DIR.rstrip('/') + '/*.jsonl')

raw = raw.persist()
print(f'Raw DataFrame rows: {raw.count():,}')


## 7. Parse `AuditData` JSON

Schema mirrors the existing `Copilot_Audit_Log_Parser.ipynb` — only the fields the PBIT consumes.


In [ ]:
audit_schema = StructType([
    StructField('CreationTime', StringType()),
    StructField('UserId', StringType()),
    StructField('Workload', StringType()),
    StructField('ApplicationName', StringType()),
    StructField('ClientRegion', StringType()),
    StructField('AgentId', StringType()),
    StructField('AgentName', StringType()),
    StructField('AppIdentity', StructType([
        StructField('AppId', StringType()),
        StructField('DisplayName', StringType()),
        StructField('PublisherId', StringType()),
    ])),
    StructField('CopilotEventData', StructType([
        StructField('AppHost', StringType()),
        StructField('ThreadId', StringType()),
        StructField('SensitivityLabelId', StringType()),
        StructField('Contexts', ArrayType(StructType([
            StructField('Type', StringType())
        ]))),
        StructField('AISystemPlugin', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('Name', StringType())
        ]))),
        StructField('ModelTransparencyDetails', ArrayType(StructType([
            StructField('ModelName', StringType())
        ]))),
        StructField('AccessedResources', ArrayType(StructType([
            StructField('Type', StringType()),
            StructField('Action', StringType()),
            StructField('SiteUrl', StringType()),
            StructField('SensitivityLabelId', StringType()),
        ]))),
        StructField('Messages', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('isPrompt', StringType())
        ]))),
    ])),
])

parsed = (raw
    .filter(F.col('AuditData').isNotNull() & (F.length('AuditData') > 10))
    .withColumn('j', F.from_json('AuditData', audit_schema))
    .filter(F.col('j').isNotNull()))


## 8. Flatten + fan out per (prompt × resource)

Mirrors the M-query exactly: drop responses, expand resources. One row per prompt-resource pair.


In [ ]:
flat = (
    parsed
    .select(
        F.col('j.CreationTime').cast('timestamp').alias('CreationDate'),
        F.col('j.AgentId').alias('AgentId'),
        F.col('j.AgentName').alias('AgentName'),
        F.col('j.AppIdentity.AppId').alias('AppIdentity_AppId'),
        F.col('j.AppIdentity.DisplayName').alias('AppIdentity_DisplayName'),
        F.col('j.AppIdentity.PublisherId').alias('AppIdentity_PublisherId'),
        F.col('j.ApplicationName').alias('ApplicationName'),
        F.col('j.ClientRegion').alias('ClientRegion'),
        F.col('j.UserId').alias('Audit_UserId'),
        F.lower(F.trim(F.col('j.UserId'))).alias('Audit_UserId_Normalized'),
        F.col('j.Workload').alias('Workload'),
        F.col('j.CopilotEventData.AppHost').alias('AppHost'),
        F.col('j.CopilotEventData.ThreadId').alias('ThreadId'),
        F.col('j.CopilotEventData.SensitivityLabelId').alias('SensitivityLabelId'),
        F.col('j.CopilotEventData.Contexts')[0]['Type'].alias('Context_Type'),
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Id'].alias('AISystemPlugin_Id'),
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Name'].alias('AISystemPlugin_Name'),
        F.col('j.CopilotEventData.ModelTransparencyDetails')[0]['ModelName']
            .alias('ModelTransparencyDetails_ModelName'),
        F.col('j.CopilotEventData.AccessedResources').alias('Resources'),
        F.col('j.CopilotEventData.Messages').alias('Messages'),
        F.size('j.CopilotEventData.AccessedResources').alias('Resource_Count'),
    )
    .withColumn('msg', F.explode('Messages'))
    .filter(F.lower(F.col('msg.isPrompt').cast('string')) == 'true')
    .withColumn(
        'res',
        F.explode_outer(
            F.when(F.size('Resources') > 0, F.col('Resources'))
             .otherwise(F.array(F.struct(
                 F.lit(None).cast(StringType()).alias('Type'),
                 F.lit(None).cast(StringType()).alias('Action'),
                 F.lit(None).cast(StringType()).alias('SiteUrl'),
                 F.lit(None).cast(StringType()).alias('SensitivityLabelId'),
             )))
        ),
    )
    .select(
        'CreationDate', 'AgentId', 'AgentName',
        'AppIdentity_AppId', 'AppIdentity_DisplayName', 'AppIdentity_PublisherId',
        'ApplicationName', 'ClientRegion',
        'Audit_UserId', 'Audit_UserId_Normalized', 'Workload',
        'AppHost', 'ThreadId', 'SensitivityLabelId',
        'Context_Type',
        'AISystemPlugin_Id', 'AISystemPlugin_Name',
        'ModelTransparencyDetails_ModelName',
        F.col('res.Type').alias('AccessedResource_Type'),
        F.col('res.Action').alias('AccessedResource_Action'),
        F.col('res.SiteUrl').alias('AccessedResource_SiteUrl'),
        F.col('res.SensitivityLabelId').alias('AccessedResource_SensitivityLabelId'),
        F.col('msg.Id').alias('Message_Id'),
        F.col('msg.isPrompt').alias('Message_isPrompt'),
        F.coalesce(F.col('Resource_Count'), F.lit(1)).alias('Resource_Count'),
        F.to_date('CreationDate').alias('InteractionDate'),
        F.expr("date_trunc('week', CreationDate)").cast('date').alias('WeekStart'),
        F.expr("date_trunc('month', CreationDate)").cast('date').alias('MonthStart'),
    )
)


## 9. Derive `Agent_TitleID`

Same logic as the existing parser + M-query.


In [ ]:
flat = flat.withColumn('Agent_TitleID', F.expr(r'''
    CASE
      WHEN AgentId IS NULL THEN NULL
      WHEN AgentId LIKE '%CopilotStudio.Declarative.%' THEN
           split(split(AgentId, 'CopilotStudio\\.Declarative\\.')[1], '\\.')[0]
      WHEN AgentId LIKE 'P\\_%' OR AgentId LIKE 'T\\_%' THEN
           split(AgentId, '\\.')[0]
      ELSE NULL
    END
'''))


## 10. Write to Lakehouse Delta table


In [ ]:
(flat.write
    .format('delta')
    .mode(WRITE_MODE)
    .option('overwriteSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))

row_count = spark.table(OUTPUT_TABLE).count()
print(f'✓ Rows written to {OUTPUT_TABLE}: {row_count:,}')


## 11. Verify

Spot-check the output. Expect populated `AISystemPlugin_Id` (e.g. `BingWebSearch`), `Workload`, `AppHost`.


In [ ]:
spark.table(OUTPUT_TABLE).select(
    'AISystemPlugin_Id', 'Workload', 'AppHost'
).groupBy('AISystemPlugin_Id', 'Workload', 'AppHost').count().orderBy(F.desc('count')).show(20, truncate=False)


---
**Connect the PBIT**: open the Fabric variant of either dashboard and supply the **Fabric SQL Endpoint** + **Lakehouse Database** parameters as before. The PBIT reads `Copilot_Interactions_Parsed` directly via the SQL endpoint — no other parameter changes needed when switching from the CSV-based parser to this direct ingester.
